In [ ]:
import os
import shutil
from collections import defaultdict

# ✅ Update this path to your test dataset
test_dataset_path = "/content/drive/MyDrive/test"
image_dir = os.path.join(test_dataset_path, "images")
label_dir = os.path.join(test_dataset_path, "labels")

# ✅ Output path where class-wise sorted data will be saved
output_base = os.path.join(test_dataset_path, "classwise_sorted")
os.makedirs(output_base, exist_ok=True)

# ✅ Mapping of class indices to class names
class_map = {
    0: "Formicidae",
    1: "Arachnida",
    2: "Coleoptera",
    3: "Nematocera",
    4: "Apoidea",
    5: "Syraphidae",
    6: "Brachycera"
}

# ✅ Create folders for each class and background
for class_name in class_map.values():
    os.makedirs(os.path.join(output_base, class_name, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_base, class_name, "labels"), exist_ok=True)
os.makedirs(os.path.join(output_base, "background", "images"), exist_ok=True)

# ✅ Track image counts
class_counts = defaultdict(int)
background_count = 0

# ✅ Process each image
for img_file in os.listdir(image_dir):
    if not img_file.endswith(".jpg"):
        continue
    image_path = os.path.join(image_dir, img_file)
    label_path = os.path.join(label_dir, img_file.replace(".jpg", ".txt"))

    if not os.path.exists(label_path) or os.path.getsize(label_path) == 0:
        # Treat as background
        shutil.copy(image_path, os.path.join(output_base, "background", "images", img_file))
        background_count += 1
        continue

    with open(label_path, 'r') as f:
        lines = f.readlines()
        class_ids = [int(line.split()[0]) for line in lines if line.strip()]
        if not class_ids:
            # Empty label file
            shutil.copy(image_path, os.path.join(output_base, "background", "images", img_file))
            background_count += 1
            continue
        # Pick the majority class in the image (or just the first)
        primary_class_id = class_ids[0]
        class_name = class_map.get(primary_class_id, "unknown")

        dst_img = os.path.join(output_base, class_name, "images", img_file)
        dst_lbl = os.path.join(output_base, class_name, "labels", img_file.replace(".jpg", ".txt"))

        shutil.copy(image_path, dst_img)
        shutil.copy(label_path, dst_lbl)
        class_counts[class_name] += 1

# ✅ Print final class-wise image count
print("\n📊 Image count per class:")
for class_name in sorted(class_counts.keys()):
    print(f"  {class_name}: {class_counts[class_name]} images")
print(f"\n🟫 Background: {background_count} images")



📊 Image count per class:
  Apoidea: 17 images
  Arachnida: 474 images
  Brachycera: 5 images
  Coleoptera: 260 images
  Formicidae: 368 images
  Nematocera: 245 images
  Syraphidae: 5 images

🟫 Background: 39 images


In [ ]:
import os
import shutil

# ✅ Path where classwise folders (with 'images' and 'labels' subfolders) exist
classwise_root = "/content/drive/MyDrive/test/classwise_sorted"

# ✅ Output merged dataset directory
merged_dir = os.path.join(classwise_root, "merged_dataset")
merged_images_dir = os.path.join(merged_dir, "images")
merged_labels_dir = os.path.join(merged_dir, "labels")
os.makedirs(merged_images_dir, exist_ok=True)
os.makedirs(merged_labels_dir, exist_ok=True)

# ✅ Loop through class folders and merge valid image-label pairs
merged_count = 0
skipped_count = 0

for class_folder in os.listdir(classwise_root):
    class_path = os.path.join(classwise_root, class_folder)
    if not os.path.isdir(class_path) or class_folder == "merged_dataset":
        continue

    images_path = os.path.join(class_path, "images")
    labels_path = os.path.join(class_path, "labels")

    if not os.path.exists(images_path) or not os.path.exists(labels_path):
        print(f"⚠️ Skipping {class_folder} — missing 'images/' or 'labels/'")
        continue

    for img_file in os.listdir(images_path):
        if not img_file.endswith(".jpg"):
            continue

        label_file = img_file.replace(".jpg", ".txt")
        src_img = os.path.join(images_path, img_file)
        src_lbl = os.path.join(labels_path, label_file)

        if os.path.exists(src_lbl):
            shutil.copy(src_img, os.path.join(merged_images_dir, img_file))
            shutil.copy(src_lbl, os.path.join(merged_labels_dir, label_file))
            merged_count += 1
        else:
            skipped_count += 1

print(f"\n✅ Merged {merged_count} image-label pairs.")
print(f"⚠️ Skipped {skipped_count} images without labels.")
print(f"📁 Merged dataset located at: {merged_dir}")


⚠️ Skipping background — missing 'images/' or 'labels/'

✅ Merged 1231 image-label pairs.
⚠️ Skipped 0 images without labels.
📁 Merged dataset located at: /content/drive/MyDrive/test/classwise_sorted/merged_dataset


In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import os, cv2
import numpy as np
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score

# ✅ Configurations
model_paths = {
    "model_1": "/content/drive/MyDrive/best_paul.pt",
    "model_2": "/content/drive/MyDrive/YOLOv11_Results/reindexed_with_background_1024_curated_corrected_again/train_yolo11n_reindexed_with_background_1000_curated_corrected_again2/weights/best.pt"
}

test_images_dir = "/content/drive/MyDrive/test/classwise_sorted/merged_dataset/images"
test_labels_dir = "/content/drive/MyDrive/test/classwise_sorted/merged_dataset/labels"

save_dir = "/content/drive/MyDrive/YOLOv11_Test_Evaluation"
os.makedirs(save_dir, exist_ok=True)

class_names = {
    0: "Formicidae",
    1: "Arachnida",
    2: "Coleoptera",
    3: "Nematocera",
    4: "Apoidea",
    5: "Syrphidae",
    6: "Brachycera"
}

# ✅ IOU helper
def compute_iou(box1, box2):
    xA, yA = max(box1[0], box2[0]), max(box1[1], box2[1])
    xB, yB = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (area1 + area2 - inter + 1e-6)

# ✅ Main evaluation loop
for model_name, path in model_paths.items():
    print(f"\n📦 Evaluating: {model_name}")
    model = YOLO(path)

    model_fp = os.path.join(save_dir, model_name, "false_positives")
    model_fn = os.path.join(save_dir, model_name, "false_negatives")
    model_mc = os.path.join(save_dir, model_name, "misclassified")
    os.makedirs(model_fp, exist_ok=True)
    os.makedirs(model_fn, exist_ok=True)
    os.makedirs(model_mc, exist_ok=True)

    y_true, y_pred = [], []

    for img_name in tqdm(os.listdir(test_images_dir)):
        if not img_name.endswith(".jpg"):
            continue

        img_path = os.path.join(test_images_dir, img_name)
        label_path = os.path.join(test_labels_dir, img_name.replace(".jpg", ".txt"))
        image = cv2.imread(img_path)
        height, width = image.shape[:2]

        result = model(img_path, conf=0.25, iou=0.7)[0]
        preds = result.boxes
        pred_boxes = preds.xyxy.cpu().numpy()
        pred_classes = preds.cls.cpu().numpy()

        gt_boxes, gt_classes = [], []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    cls, x, y, w, h = map(float, line.strip().split())
                    x1 = int((x - w/2) * width)
                    y1 = int((y - h/2) * height)
                    x2 = int((x + w/2) * width)
                    y2 = int((y + h/2) * height)
                    gt_boxes.append([x1, y1, x2, y2])
                    gt_classes.append(int(cls))

        matched_pred, matched_gt = set(), set()

        for i, pbox in enumerate(pred_boxes):
            px1, py1, px2, py2 = map(int, pbox[:4])
            pred_cls = int(pred_classes[i])
            for j, gtbox in enumerate(gt_boxes):
                iou = compute_iou(pbox[:4], gtbox)
                if iou > 0.3:
                    matched_pred.add(i)
                    matched_gt.add(j)
                    if pred_cls != gt_classes[j]:
                        img_copy = image.copy()
                        gx1, gy1, gx2, gy2 = gtbox
                        cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (0, 255, 0), 2)
                        cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
                        cv2.putText(img_copy, f"Pred: {class_names.get(pred_cls)}", (px1, py1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
                        cv2.putText(img_copy, f"GT: {class_names.get(gt_classes[j])}", (gx1, gy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
                        cv2.imwrite(os.path.join(model_mc, f"mc_{img_name}"), img_copy)
                    break

        for i, pbox in enumerate(pred_boxes):
            if i not in matched_pred:
                y_true.append(0)
                y_pred.append(1)
                px1, py1, px2, py2 = map(int, pbox[:4])
                pred_cls = int(pred_classes[i])
                img_copy = image.copy()
                cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
                cv2.putText(img_copy, f"FP: {class_names.get(pred_cls)}", (px1, py1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
                cv2.imwrite(os.path.join(model_fp, f"fp_{img_name}"), img_copy)

        for j, gtbox in enumerate(gt_boxes):
            if j not in matched_gt:
                y_true.append(1)
                y_pred.append(0)
                gx1, gy1, gx2, gy2 = gtbox
                img_copy = image.copy()
                cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (255, 0, 0), 2)
                cv2.putText(img_copy, "FN", (gx1, gy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
                cv2.imwrite(os.path.join(model_fn, f"fn_{img_name}"), img_copy)

    # ✅ Save results
    precision = precision_score(y_true, y_pred) * 100
    recall = recall_score(y_true, y_pred) * 100
    f1 = f1_score(y_true, y_pred) * 100

    print(f"\n📊 Evaluation for {model_name}")
    print(f"Precision: {precision:.2f}%")
    print(f"Recall:    {recall:.2f}%")
    print(f"F1 Score:  {f1:.2f}%")
    print(f"FP saved to: {model_fp}")
    print(f"FN saved to: {model_fn}")
    print(f"MC saved to: {model_mc}")
    from sklearn.metrics import classification_report

    # Get per-class ground truth and predictions
    per_class_y_true = []
    per_class_y_pred = []

    for i in matched_pred:
        per_class_y_pred.append(int(pred_classes[i]))
    for j in matched_gt:
        per_class_y_true.append(int(gt_classes[j]))

    # Combine and get unique labels for report
    unique_labels = sorted(set(per_class_y_true + per_class_y_pred))
    target_names = [class_names[i] for i in unique_labels]

    # Classification report
    report = classification_report(
        per_class_y_true,
        per_class_y_pred,
        labels=unique_labels,
        target_names=target_names,
        digits=3,
        zero_division=0
    )

    print(f"\n📋 Classification Report for {model_name}:\n{report}")

    # Save report to file
    report_path = os.path.join(save_dir, model_name, "classification_report.txt")
    with open(report_path, "w") as f:
        f.write(report)
    print(f"📄 Report saved to: {report_path}")





📦 Evaluating: model_1


  0%|          | 0/1231 [00:00<?, ?it/s]


image 1/1 /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/859_2387667.jpg: 1024x1024 1 Formicidae, 14.0ms
Speed: 7.1ms preprocess, 14.0ms inference, 1.5ms postprocess per image at shape (1, 3, 1024, 1024)


  0%|          | 1/1231 [00:00<04:35,  4.47it/s]


image 1/1 /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/859_2079834.jpg: 1024x1024 1 Formicidae, 14.1ms
Speed: 6.3ms preprocess, 14.1ms inference, 1.2ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/859_2313961.jpg: 1024x1024 2 Formicidaes, 14.1ms
Speed: 6.6ms preprocess, 14.1ms inference, 1.4ms postprocess per image at shape (1, 3, 1024, 1024)


...


image 1/1 /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/859_2179590.jpg: 1024x1024 1 Formicidae, 15.0ms
Speed: 10.1ms preprocess, 15.0ms inference, 1.7ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/859_2280945.jpg: 1024x1024 1 Formicidae, 14.2ms
Speed: 9.6ms preprocess, 14.2ms inference, 1.3ms postprocess per image at shape (1, 3, 1024, 1024)


100%|█████████▉| 1230/1231 [03:02<00:00,  6.04it/s]


image 1/1 /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/859_2105507.jpg: 1024x1024 1 Formicidae, 21.6ms
Speed: 10.6ms preprocess, 21.6ms inference, 1.7ms postprocess per image at shape (1, 3, 1024, 1024)


100%|██████████| 1231/1231 [03:02<00:00,  6.76it/s]


📊 Evaluation for model_2
Precision: 0.00%
Recall:    0.00%
F1 Score:  0.00%
FP saved to: /content/drive/MyDrive/YOLOv11_Test_Evaluation/model_2/false_positives
FN saved to: /content/drive/MyDrive/YOLOv11_Test_Evaluation/model_2/false_negatives
MC saved to: /content/drive/MyDrive/YOLOv11_Test_Evaluation/model_2/misclassified

📋 Classification Report for model_2:
              precision    recall  f1-score   support

  Formicidae      0.000     0.000     0.000       1.0
     Apoidea      0.000     0.000     0.000       0.0

    accuracy                          0.000       1.0
   macro avg      0.000     0.000     0.000       1.0
weighted avg      0.000     0.000     0.000       1.0

📄 Report saved to: /content/drive/MyDrive/YOLOv11_Test_Evaluation/model_2/classification_report.txt


In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import os

# Paths to models and configs
model_configs = {
    "model_1": {
        "path": "/content/drive/MyDrive/best_paul.pt",
        "config": "/content/drive/MyDrive/test/classwise_sorted/merged_dataset/config_paul.yaml"
    },
    "model_2": {
        "path": "/content/drive/MyDrive/YOLOv11_Results/reindexed_with_background_1024_curated_corrected_again/train_yolo11n_reindexed_with_background_1000_curated_corrected_again2/weights/best.pt",
        "config": "/content/drive/MyDrive/test/classwise_sorted/merged_dataset/config_mine.yaml"
    }
}

# Where to save results
save_dir = "/content/drive/MyDrive/YOLOv11_Test_ObjectDetection_Reports"
os.makedirs(save_dir, exist_ok=True)

# Run validation and save results for each model
for name, cfg in model_configs.items():
    print(f"\n🔍 Running val() for {name}")
    model = YOLO(cfg["path"])

    results = model.val(
        data=cfg["config"],
        save=True,
        save_json=True,
        project=os.path.join(save_dir, name),
        name="val_run",
        conf=0.25,
        iou=0.7,
        imgsz=1000
    )

    print(f"📊 Report for {name} saved to: {os.path.join(save_dir, name)}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.8/974.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 62.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

100%|██████████| 755k/755k [00:00<00:00, 15.1MB/s]
val: Scanning /content/drive/MyDrive/test/classwise_sorted/merged_dataset/labels... 1231 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1231/1231 [05:27<00:00,  3.76it/s]

val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/562_2123793.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/562_2123934.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/562_2123951.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/562_2173789.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/562_2175219.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/793_2108902.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/793_2123613.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/793_2135266.jpg: 1 duplicate

val: New cache created: /content/drive/MyDrive/test/classwise_sorted/merged_dataset/labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 77/77 [00:43<00:00,  1.79it/s]


                   all       1231       1341     0.0982     0.0798     0.0723     0.0557
            Syraphidae        298        353      0.125    0.00283     0.0671     0.0603
            Nematocera        441        461      0.208     0.0564       0.11     0.0965
               Apoidea        268        268          0          0          0          0
            Formicidae        218        232      0.069      0.147     0.0397     0.0282
            Brachycera         17         17      0.286      0.353      0.289      0.205
            Coleoptera          5          5          0          0          0          0
             Arachnida          5          5          0          0          0          0
Speed: 1.1ms preprocess, 9.9ms inference, 0.0ms loss, 2.0ms postprocess per image
Saving /content/drive/MyDrive/YOLOv11_Test_ObjectDetection_Reports/model_1/val_run/predictions.json...
Results saved to /content/drive/MyDrive/YOLOv11_Test_ObjectDetection_Reports/model_1/val_run
📊 Report f

val: Scanning /content/drive/MyDrive/test/classwise_sorted/merged_dataset/labels.cache... 1231 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1231/1231 [00:00<?, ?it/s]

val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/562_2123793.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/562_2123934.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/562_2123951.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/562_2173789.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/562_2175219.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/793_2108902.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/793_2123613.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/classwise_sorted/merged_dataset/images/793_2135266.jpg: 1 duplicate


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 77/77 [00:48<00:00,  1.60it/s]


                   all       1231       1341     0.0863      0.066     0.0706     0.0467
               Apoidea        298        353          0          0          0          0
             Arachnida        441        461       0.42      0.334       0.39       0.27
            Brachycera        268        268     0.0169    0.00746    0.00857    0.00343
            Coleoptera        218        232      0.168      0.121     0.0953     0.0531
            Formicidae         17         17          0          0          0          0
            Nematocera          5          5          0          0          0          0
            Syraphidae          5          5          0          0          0          0
Speed: 1.4ms preprocess, 9.2ms inference, 0.0ms loss, 1.4ms postprocess per image
Saving /content/drive/MyDrive/YOLOv11_Test_ObjectDetection_Reports/model_2/val_run/predictions.json...
Results saved to /content/drive/MyDrive/YOLOv11_Test_ObjectDetection_Reports/model_2/val_run
📊 Report f

In [ ]:
import os

labels_dir = "/content/drive/MyDrive/test/classwise_sorted/merged_dataset/labels"
duplicates_found = []

for fname in os.listdir(labels_dir):
    if not fname.endswith(".txt"):
        continue
    fpath = os.path.join(labels_dir, fname)
    with open(fpath, 'r') as f:
        lines = f.readlines()
    if len(lines) != len(set(lines)):
        duplicates_found.append(fname)

print(f"🧹 Found {len(duplicates_found)} label files with duplicate entries.")
print("Examples:", duplicates_found[:5])


🧹 Found 0 label files with duplicate entries.
Examples: []


In [ ]:
image_dir = "/content/drive/MyDrive/test/classwise_sorted/merged_dataset/images"
label_dir = "/content/drive/MyDrive/test/classwise_sorted/merged_dataset/labels"

backgrounds = []
for fname in os.listdir(image_dir):
    if fname.endswith(".jpg"):
        label_path = os.path.join(label_dir, fname.replace(".jpg", ".txt"))
        if not os.path.exists(label_path):
            backgrounds.append(fname)

print(f"✅ Found {len(backgrounds)} true background images.")


✅ Found 0 true background images.


In [ ]:
import os

# Set your label directory
label_dir = "/content/drive/MyDrive/test/classwise_sorted/merged_dataset/labels"

# Count of removed files
empty_labels_removed = 0

# Loop and remove empty .txt files
for fname in os.listdir(label_dir):
    if fname.endswith(".txt"):
        fpath = os.path.join(label_dir, fname)
        if os.path.getsize(fpath) == 0:  # Empty file
            os.remove(fpath)
            empty_labels_removed += 1

print(f"🧹 Removed {empty_labels_removed} empty label files (backgrounds).")


🧹 Removed 0 empty label files (backgrounds).


In [ ]:
import os
from collections import Counter

# ✅ Update paths
labels_dir = "/content/drive/MyDrive/test/classwise_sorted/merged_dataset/labels"
images_dir = "/content/drive/MyDrive/test/classwise_sorted/merged_dataset/images"

# Configuration mapping (update if needed)
class_names = {
    0: "Formicidae",
    1: "Arachnida",
    2: "Coleoptera",
    3: "Nematocera",
    4: "Apoidea",
    5: "Syrphidae",
    6: "Brachycera"
}

# ✅ Count instances per class
class_counts = Counter()
for label_file in os.listdir(labels_dir):
    if not label_file.endswith(".txt"):
        continue
    with open(os.path.join(labels_dir, label_file)) as f:
        for line in f:
            if line.strip():  # non-empty line
                class_id = int(line.strip().split()[0])
                class_counts[class_id] += 1

# 🔍 Display class-wise distribution
print("📊 Class Distribution:")
for cls_id in sorted(class_counts):
    print(f"  Class {cls_id} ({class_names.get(cls_id, 'Unknown')}): {class_counts[cls_id]} instances")

# 🚫 Find image files with no corresponding label
image_files = [f for f in os.listdir(images_dir) if f.endswith(".jpg")]
missing_labels = []
for img_file in image_files:
    label_file = img_file.replace(".jpg", ".txt")
    if not os.path.exists(os.path.join(labels_dir, label_file)):
        missing_labels.append(img_file)

# 🔍 Show results
print(f"\n🖼️ Total images: {len(image_files)}")
print(f"❌ Images with no label file: {len(missing_labels)}")
if missing_labels:
    print("Examples:")
    print("\n".join(missing_labels[:10]))  # show first 10


📊 Class Distribution:
  Class 0 (Formicidae): 373 instances
  Class 1 (Arachnida): 463 instances
  Class 2 (Coleoptera): 269 instances
  Class 3 (Nematocera): 239 instances
  Class 4 (Apoidea): 17 instances
  Class 5 (Syrphidae): 5 instances
  Class 6 (Brachycera): 5 instances

🖼️ Total images: 1231
❌ Images with no label file: 0


In [ ]:
import os
from collections import Counter

# ✅ Update paths
labels_dir = "/content/drive/MyDrive/test/labels"
images_dir = "/content/drive/MyDrive/test/images"

# Configuration mapping (update if needed)
class_names = {
    0: "Formicidae",
    1: "Arachnida",
    2: "Coleoptera",
    3: "Nematocera",
    4: "Apoidea",
    5: "Syrphidae",
    6: "Brachycera"
}

# ✅ Count instances per class
class_counts = Counter()
for label_file in os.listdir(labels_dir):
    if not label_file.endswith(".txt"):
        continue
    with open(os.path.join(labels_dir, label_file)) as f:
        for line in f:
            if line.strip():  # non-empty line
                class_id = int(line.strip().split()[0])
                class_counts[class_id] += 1

# 🔍 Display class-wise distribution
print("📊 Class Distribution:")
for cls_id in sorted(class_counts):
    print(f"  Class {cls_id} ({class_names.get(cls_id, 'Unknown')}): {class_counts[cls_id]} instances")

# 🚫 Find image files with no corresponding label
image_files = [f for f in os.listdir(images_dir) if f.endswith(".jpg")]
missing_labels = []
for img_file in image_files:
    label_file = img_file.replace(".jpg", ".txt")
    if not os.path.exists(os.path.join(labels_dir, label_file)):
        missing_labels.append(img_file)

# 🔍 Show results
print(f"\n🖼️ Total images: {len(image_files)}")
print(f"❌ Images with no label file: {len(missing_labels)}")
if missing_labels:
    print("Examples:")
    print("\n".join(missing_labels[:10]))  # show first 10


📊 Class Distribution:
  Class 0 (Formicidae): 471 instances
  Class 1 (Arachnida): 507 instances
  Class 2 (Coleoptera): 272 instances
  Class 3 (Nematocera): 304 instances
  Class 4 (Apoidea): 17 instances
  Class 5 (Syrphidae): 5 instances
  Class 6 (Brachycera): 5 instances

🖼️ Total images: 1413
❌ Images with no label file: 39
Examples:
2025-04-11_09-46-03-384826_pos4.45_rate4_imx708_0.jpg.jpg
2025-04-11_09-46-02-094727_pos4.45_rate4_imx708_0.jpg.jpg
2025-04-11_09-42-23-937069_pos4.45_rate4_imx708_0.jpg.jpg
2025-04-11_09-40-43-561826_pos4.45_rate4_imx708_0.jpg.jpg
2025-04-11_09-34-05-963855_pos4.45_rate4_imx708_0.jpg.jpg
2025-04-11_09-23-18-768796_pos4.45_rate4_imx708_0.jpg.jpg
2025-04-11_09-23-07-931572_pos4.45_rate4_imx708_0.jpg.jpg
2025-04-11_09-49-21-492660_pos4.45_rate4_imx708_0.jpg.jpg
2025-04-11_10-18-38-832025_pos4.45_rate4_imx708_0.jpg.jpg
2025-04-11_10-16-15-305769_pos4.45_rate4_imx708_0.jpg.jpg


In [ ]:
import os
from collections import Counter

# ✅ Update paths
labels_dir = "/content/drive/MyDrive/test/classwise_sorted/Formicidae/labels"
images_dir = "/content/drive/MyDrive/test/classwise_sorted/Formicidae/images"

# Configuration mapping (update if needed)
class_names = {
    0: "Formicidae",
    1: "Arachnida",
    2: "Coleoptera",
    3: "Nematocera",
    4: "Apoidea",
    5: "Syrphidae",
    6: "Brachycera"
}

# ✅ Count instances per class
class_counts = Counter()
for label_file in os.listdir(labels_dir):
    if not label_file.endswith(".txt"):
        continue
    with open(os.path.join(labels_dir, label_file)) as f:
        for line in f:
            if line.strip():  # non-empty line
                class_id = int(line.strip().split()[0])
                class_counts[class_id] += 1

# 🔍 Display class-wise distribution
print("📊 Class Distribution:")
for cls_id in sorted(class_counts):
    print(f"  Class {cls_id} ({class_names.get(cls_id, 'Unknown')}): {class_counts[cls_id]} instances")

# 🚫 Find image files with no corresponding label
image_files = [f for f in os.listdir(images_dir) if f.endswith(".jpg")]
missing_labels = []
for img_file in image_files:
    label_file = img_file.replace(".jpg", ".txt")
    if not os.path.exists(os.path.join(labels_dir, label_file)):
        missing_labels.append(img_file)

# 🔍 Show results
print(f"\n🖼️ Total images: {len(image_files)}")
print(f"❌ Images with no label file: {len(missing_labels)}")
if missing_labels:
    print("Examples:")
    print("\n".join(missing_labels[:10]))  # show first 10


📊 Class Distribution:
  Class 0 (Formicidae): 471 instances

🖼️ Total images: 298
❌ Images with no label file: 0


In [ ]:
import os
import shutil
from collections import defaultdict

# ✅ Paths
source_root = "/content/drive/MyDrive/test/classwise_sorted"
merged_dir = "/content/drive/MyDrive/test/merged_dataset_again"
merged_images_dir = os.path.join(merged_dir, "images")
merged_labels_dir = os.path.join(merged_dir, "labels")
os.makedirs(merged_images_dir, exist_ok=True)
os.makedirs(merged_labels_dir, exist_ok=True)

# ✅ Define the class folders (excluding background)
all_folders = sorted(os.listdir(source_root))
class_folders = [f for f in all_folders if f.lower() != "background"]

# ✅ Class count tracker
class_image_counts = defaultdict(int)
total_merged = 0
missing_labels = []

# ✅ Merge each class folder
for class_name in class_folders:
    img_dir = os.path.join(source_root, class_name, "images")
    lbl_dir = os.path.join(source_root, class_name, "labels")
    if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
        print(f"⚠️ Skipped {class_name} (missing 'images' or 'labels' folder)")
        continue

    for fname in os.listdir(img_dir):
        if not fname.endswith(".jpg"):
            continue
        label_fname = fname.replace(".jpg", ".txt")
        img_path = os.path.join(img_dir, fname)
        lbl_path = os.path.join(lbl_dir, label_fname)

        if not os.path.exists(lbl_path):
            missing_labels.append(fname)
            continue

        # Copy to merged directory
        shutil.copy(img_path, os.path.join(merged_images_dir, fname))
        shutil.copy(lbl_path, os.path.join(merged_labels_dir, label_fname))
        class_image_counts[class_name] += 1
        total_merged += 1

# ✅ Optionally: Add background-only images
background_dir = os.path.join(source_root, "background", "images")
background_added = 0
if os.path.exists(background_dir):
    for fname in os.listdir(background_dir):
        if fname.endswith(".jpg"):
            shutil.copy(os.path.join(background_dir, fname), os.path.join(merged_images_dir, fname))
            background_added += 1

# ✅ Summary
print("\n📊 Merge Summary:")
for cls, count in class_image_counts.items():
    print(f"  🔸 {cls}: {count} images")

print(f"\n✅ Total image-label pairs merged: {total_merged}")
print(f"✅ Background-only images added: {background_added}")
print(f"❌ Missing label files for {len(missing_labels)} images")

if missing_labels:
    print("\n📝 Some images were skipped due to missing labels (showing up to 5):")
    print(missing_labels[:5])



📊 Merge Summary:
  🔸 Apoidea: 17 images
  🔸 Arachnida: 431 images
  🔸 Brachycera: 5 images
  🔸 Coleoptera: 257 images
  🔸 Formicidae: 298 images
  🔸 Nematocera: 218 images
  🔸 Syraphidae: 5 images
  🔸 merged_dataset: 1231 images

✅ Total image-label pairs merged: 2462
✅ Background-only images added: 39
❌ Missing label files for 0 images


In [ ]:
import os
import shutil
from tqdm import tqdm

# ✅ Source and destination
source_root = "/content/drive/MyDrive/test/classwise_sorted"
output_root = "/content/drive/MyDrive/test_reindexed_for_colleague_again"

# ✅ Mapping: your model class name → colleague’s index
class_name_to_new_index = {
    "formicidae": 3,
    "arachnida": 6,
    "coleoptera": 5,
    "nematocera": 1,
    "apoidea": 2,
    "syraphidae": 0,
    "brachycera": 4
}

# ✅ Create destination folders
merged_images = os.path.join(output_root, "images")
merged_labels = os.path.join(output_root, "labels")
os.makedirs(merged_images, exist_ok=True)
os.makedirs(merged_labels, exist_ok=True)

# ✅ Summary dictionary
summary = {}

for class_name in tqdm(os.listdir(source_root), desc="🔁 Processing Classes"):
    class_path = os.path.join(source_root, class_name)
    class_name_lower = class_name.lower()

    # Handle background folder
    if class_name_lower == "background":
        bg_img_dir = os.path.join(class_path, "images")
        bg_count = 0
        for img_file in os.listdir(bg_img_dir):
            if img_file.endswith(".jpg"):
                shutil.copy(os.path.join(bg_img_dir, img_file), os.path.join(merged_images, img_file))
                bg_count += 1
        summary["background"] = bg_count
        continue

    # Handle class folders
    if class_name_lower not in class_name_to_new_index:
        print(f"⚠️ Skipped: Unmapped class folder -> {class_name}")
        continue

    new_class_index = class_name_to_new_index[class_name_lower]
    img_dir = os.path.join(class_path, "images")
    lbl_dir = os.path.join(class_path, "labels")
    count = 0

    for img_file in os.listdir(img_dir):
        if not img_file.endswith(".jpg"):
            continue

        label_file = img_file.replace(".jpg", ".txt")
        src_img = os.path.join(img_dir, img_file)
        src_lbl = os.path.join(lbl_dir, label_file)

        dst_img = os.path.join(merged_images, img_file)
        dst_lbl = os.path.join(merged_labels, label_file)

        shutil.copy(src_img, dst_img)

        if os.path.exists(src_lbl):
            with open(src_lbl, "r") as f:
                lines = f.readlines()

            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) == 5:
                    parts[0] = str(new_class_index)
                    new_lines.append(" ".join(parts))

            with open(dst_lbl, "w") as f:
                f.write("\n".join(new_lines))

            count += 1

    summary[class_name] = count

# ✅ Print summary
print("\n📊 Class-wise image-label counts after reindexing:")
for cls, count in summary.items():
    print(f"{cls}: {count} images")
print(f"\n✅ Reindexed dataset saved to: {output_root}")


🔁 Processing Classes: 100%|██████████| 9/9 [00:41<00:00,  4.64s/it]

⚠️ Skipped: Unmapped class folder -> merged_dataset

📊 Class-wise image-label counts after reindexing:
Formicidae: 298 images
Arachnida: 431 images
Coleoptera: 257 images
Nematocera: 218 images
Apoidea: 17 images
Syraphidae: 5 images
Brachycera: 5 images
background: 39 images

✅ Reindexed dataset saved to: /content/drive/MyDrive/test_reindexed_for_colleague_again


In [ ]:
import os
import shutil
from tqdm import tqdm

# ✅ Source and destination
source_root = "/content/drive/MyDrive/test/classwise_sorted"
output_root = "/content/drive/MyDrive/test_reindexed_for_myself_again"

# ✅ Mapping: your model class name → colleague’s index
class_name_to_new_index = {
     "formicidae": 4,
    "arachnida": 1,
    "coleoptera": 3,
    "nematocera": 5,
    "apoidea": 0,
    "syraphidae": 6,
    "brachycera": 2
}

# ✅ Create destination folders
merged_images = os.path.join(output_root, "images")
merged_labels = os.path.join(output_root, "labels")
os.makedirs(merged_images, exist_ok=True)
os.makedirs(merged_labels, exist_ok=True)

# ✅ Summary dictionary
summary = {}

for class_name in tqdm(os.listdir(source_root), desc="🔁 Processing Classes"):
    class_path = os.path.join(source_root, class_name)
    class_name_lower = class_name.lower()

    # Handle background folder
    if class_name_lower == "background":
        bg_img_dir = os.path.join(class_path, "images")
        bg_count = 0
        for img_file in os.listdir(bg_img_dir):
            if img_file.endswith(".jpg"):
                shutil.copy(os.path.join(bg_img_dir, img_file), os.path.join(merged_images, img_file))
                bg_count += 1
        summary["background"] = bg_count
        continue

    # Handle class folders
    if class_name_lower not in class_name_to_new_index:
        print(f"⚠️ Skipped: Unmapped class folder -> {class_name}")
        continue

    new_class_index = class_name_to_new_index[class_name_lower]
    img_dir = os.path.join(class_path, "images")
    lbl_dir = os.path.join(class_path, "labels")
    count = 0

    for img_file in os.listdir(img_dir):
        if not img_file.endswith(".jpg"):
            continue

        label_file = img_file.replace(".jpg", ".txt")
        src_img = os.path.join(img_dir, img_file)
        src_lbl = os.path.join(lbl_dir, label_file)

        dst_img = os.path.join(merged_images, img_file)
        dst_lbl = os.path.join(merged_labels, label_file)

        shutil.copy(src_img, dst_img)

        if os.path.exists(src_lbl):
            with open(src_lbl, "r") as f:
                lines = f.readlines()

            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) == 5:
                    parts[0] = str(new_class_index)
                    new_lines.append(" ".join(parts))

            with open(dst_lbl, "w") as f:
                f.write("\n".join(new_lines))

            count += 1

    summary[class_name] = count

# ✅ Print summary
print("\n📊 Class-wise image-label counts after reindexing:")
for cls, count in summary.items():
    print(f"{cls}: {count} images")
print(f"\n✅ Reindexed dataset saved to: {output_root}")


🔁 Processing Classes: 100%|██████████| 9/9 [05:40<00:00, 37.82s/it]

⚠️ Skipped: Unmapped class folder -> merged_dataset

📊 Class-wise image-label counts after reindexing:
Formicidae: 298 images
Arachnida: 431 images
Coleoptera: 257 images
Nematocera: 218 images
Apoidea: 17 images
Syraphidae: 5 images
Brachycera: 5 images
background: 39 images

✅ Reindexed dataset saved to: /content/drive/MyDrive/test_reindexed_for_myself_again


In [ ]:
# !pip install ultralytics
from ultralytics import YOLO
import os

# ✅ Path Configs
model_path = "/content/drive/MyDrive/YOLOv11_Results/reindexed_with_background_1024_curated_corrected_again/train_yolo11n_reindexed_with_background_1000_curated_corrected_again2/weights/best.pt"  # <-- 🔁 Update with your model file
yaml_path = "/content/drive/MyDrive/test/merged_dataset_again/my_confi.yaml"
output_dir = "/content/drive/MyDrive/YOLOv11_Test_Results_Final"  # Where to save results

# ✅ Run Validation (includes confusion matrix, PR curve, metrics)
model = YOLO(model_path)
metrics = model.val(
    data=yaml_path,
    save=True,
    save_json=True,
    project=output_dir,
    name="final_test_eval",
    imgsz=1000,  # Optional: set input size
    iou=0.7,
    conf=0.25
)

# ✅ Display results
print("\n🎯 Evaluation Metrics:")
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.precision:.4f}")
print(f"Recall:     {metrics.box.recall:.4f}")
print(f"Results saved to: {os.path.join(output_dir, 'final_test_eval')}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.8/978.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

100%|██████████| 755k/755k [00:00<00:00, 49.2MB/s]


val: Fast image access ✅ (ping: 0.5±0.0 ms, read: 1.2±0.9 MB/s, size: 500.8 KB)


val: Scanning /content/drive/MyDrive/test/merged_dataset_again/labels... 1231 images, 39 backgrounds, 0 corrupt: 100%|██████████| 1270/1270 [08:12<00:00,  2.58it/s]

val: WARNING ⚠️ /content/drive/MyDrive/test/merged_dataset_again/images/562_2123793.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/merged_dataset_again/images/562_2123934.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/merged_dataset_again/images/562_2123951.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/merged_dataset_again/images/562_2173789.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/merged_dataset_again/images/562_2175219.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/merged_dataset_again/images/793_2108902.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/merged_dataset_again/images/793_2123613.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/merged_dataset_again/images/793_2135266.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test/merged_dataset_again/images/

val: New cache created: /content/drive/MyDrive/test/merged_dataset_again/labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 80/80 [00:47<00:00,  1.68it/s]


                   all       1270       1341     0.0857      0.066     0.0701     0.0464
               Apoidea        298        353          0          0          0          0
             Arachnida        441        461      0.415      0.334      0.387      0.268
            Brachycera        268        268     0.0171    0.00746    0.00864    0.00345
            Coleoptera        218        232      0.168      0.121     0.0952      0.053
            Formicidae         17         17          0          0          0          0
            Nematocera          5          5          0          0          0          0
            Syraphidae          5          5          0          0          0          0
Speed: 1.5ms preprocess, 9.9ms inference, 0.0ms loss, 1.9ms postprocess per image
Saving /content/drive/MyDrive/YOLOv11_Test_Results_Final/final_test_eval/predictions.json...
Results saved to /content/drive/MyDrive/YOLOv11_Test_Results_Final/final_test_eval

🎯 Evaluation Metrics:
mAP50: 

AttributeError: 'Metric' object has no attribute 'precision'. See valid attributes below.

    Class for computing evaluation metrics for YOLOv8 model.

    Attributes:
        p (list): Precision for each class. Shape: (nc,).
        r (list): Recall for each class. Shape: (nc,).
        f1 (list): F1 score for each class. Shape: (nc,).
        all_ap (list): AP scores for all classes and all IoU thresholds. Shape: (nc, 10).
        ap_class_index (list): Index of class for each AP score. Shape: (nc,).
        nc (int): Number of classes.

    Methods:
        ap50(): AP at IoU threshold of 0.5 for all classes. Returns: List of AP scores. Shape: (nc,) or [].
        ap(): AP at IoU thresholds from 0.5 to 0.95 for all classes. Returns: List of AP scores. Shape: (nc,) or [].
        mp(): Mean precision of all classes. Returns: Float.
        mr(): Mean recall of all classes. Returns: Float.
        map50(): Mean AP at IoU threshold of 0.5 for all classes. Returns: Float.
        map75(

In [ ]:
# !pip install ultralytics
from ultralytics import YOLO
import os

# ✅ Path Configs
model_path = "/content/drive/MyDrive/YOLOv11_Results/reindexed_with_background_1024_curated_corrected_again/train_yolo11n_reindexed_with_background_1000_curated_corrected_again2/weights/best.pt"  # <-- 🔁 Update with your model file
yaml_path = "/content/drive/MyDrive/test_reindexed_for_myself_again/config.yaml"
output_dir = "/content/drive/MyDrive/YOLOv11_Test_Results_Final_myself"  # Where to save results

# ✅ Run Validation (includes confusion matrix, PR curve, metrics)
model = YOLO(model_path)
metrics = model.val(
    data=yaml_path,
    save=True,
    save_json=True,
    project=output_dir,
    name="final_test_eval",
    imgsz=1000,  # Optional: set input size
    iou=0.7,
    conf=0.25
)

# ✅ Display results
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")



WARNING ⚠️ imgsz=[1000] must be multiple of max stride 32, updating to [1024]
Ultralytics 8.3.110 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n summary (fused): 100 layers, 2,583,517 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.0 ms, read: 157.5±140.6 MB/s, size: 533.2 KB)


val: Scanning /content/drive/MyDrive/test_reindexed_for_myself_again/labels... 1231 images, 39 backgrounds, 0 corrupt: 100%|██████████| 1270/1270 [00:16<00:00, 75.03it/s] 

val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_myself_again/images/562_2123793.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_myself_again/images/562_2123934.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_myself_again/images/562_2123951.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_myself_again/images/562_2173789.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_myself_again/images/562_2175219.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_myself_again/images/793_2108902.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_myself_again/images/793_2123613.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_myself_again/images/793_2135266.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content

val: New cache created: /content/drive/MyDrive/test_reindexed_for_myself_again/labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 80/80 [00:49<00:00,  1.61it/s]


                   all       1270       1341      0.413      0.438      0.437      0.271
               Apoidea         17         17        0.6      0.176      0.423      0.231
             Arachnida        431        460      0.415      0.335      0.387      0.268
            Brachycera          5          5     0.0427          1       0.49      0.275
            Coleoptera        257        267      0.635      0.397      0.513      0.343
            Formicidae        298        353      0.463      0.759       0.66       0.41
            Nematocera        218        234      0.738      0.397      0.584       0.37
            Syraphidae          5          5          0          0          0          0
Speed: 1.6ms preprocess, 9.4ms inference, 0.0ms loss, 1.7ms postprocess per image
Saving /content/drive/MyDrive/YOLOv11_Test_Results_Final_myself/final_test_eval3/predictions.json...
Results saved to /content/drive/MyDrive/YOLOv11_Test_Results_Final_myself/final_test_eval3
mAP50:      0.

In [ ]:
# !pip install ultralytics
from ultralytics import YOLO
import os

# ✅ Path Configs
model_path = "/content/drive/MyDrive/best_paul.pt"  # <-- 🔁 Update with your model file
yaml_path = "/content/drive/MyDrive/test_reindexed_for_colleague_again/config.yaml"
output_dir = "/content/drive/MyDrive/YOLOv11_Test_Results_Final_paul"  # Where to save results

# ✅ Run Validation (includes confusion matrix, PR curve, metrics)
model = YOLO(model_path)
metrics = model.val(
    data=yaml_path,
    save=True,
    save_json=True,
    project=output_dir,
    name="final_test_eval",
    imgsz=1000,  # Optional: set input size
    iou=0.7,
    conf=0.25
)

# ✅ Display results
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")



WARNING ⚠️ imgsz=[1000] must be multiple of max stride 32, updating to [1024]
Ultralytics 8.3.110 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n summary (fused): 100 layers, 2,583,517 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 0.9±0.8 MB/s, size: 353.8 KB)


val: Scanning /content/drive/MyDrive/test_reindexed_for_colleague_again/labels... 1231 images, 39 backgrounds, 0 corrupt: 100%|██████████| 1270/1270 [10:58<00:00,  1.93it/s]

val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_colleague_again/images/562_2123793.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_colleague_again/images/562_2123934.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_colleague_again/images/562_2123951.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_colleague_again/images/562_2173789.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_colleague_again/images/562_2175219.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_colleague_again/images/793_2108902.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_colleague_again/images/793_2123613.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/drive/MyDrive/test_reindexed_for_colleague_again/images/793_2135266.jpg: 1 duplicate labels removed


val: New cache created: /content/drive/MyDrive/test_reindexed_for_colleague_again/labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 80/80 [00:57<00:00,  1.38it/s]


                   all       1270       1341      0.388      0.433      0.389      0.251
            Syraphidae          5          5          0          0          0          0
            Nematocera        218        234      0.525      0.312      0.428      0.291
               Apoidea         17         17       0.12      0.176     0.0755     0.0522
            Formicidae        298        353      0.507      0.711      0.599       0.37
            Brachycera          5          5      0.174        0.8      0.419      0.179
            Coleoptera        257        267      0.579      0.689      0.607      0.425
             Arachnida        431        460      0.813      0.341      0.595      0.443
Speed: 1.3ms preprocess, 9.6ms inference, 0.0ms loss, 1.8ms postprocess per image
Saving /content/drive/MyDrive/YOLOv11_Test_Results_Final_paul/final_test_eval/predictions.json...
Results saved to /content/drive/MyDrive/YOLOv11_Test_Results_Final_paul/final_test_eval
mAP50:      0.3891
m

In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import os

# ✅ Path Configs
model_path = "/content/drive/MyDrive/YOLOv11_Results/reindexed_with_background_1024_curated_corrected_again_augmentation/train_yolo11n_reindexed_with_background_1000_curated_corrected_again_augmentation2/weights/best.pt"  # <-- 🔁 Update with your model file
yaml_path = "/content/drive/MyDrive/test_reindexed_for_myself_again/config.yaml"
output_dir = "/content/drive/MyDrive/YOLOv11_Test_Results_Final_myself_augmented"  # Where to save results

# ✅ Run Validation (includes confusion matrix, PR curve, metrics)
model = YOLO(model_path)
metrics = model.val(
    data=yaml_path,
    save=True,
    save_json=True,
    project=output_dir,
    name="final_test_eval",
    imgsz=1000,  # Optional: set input size
    iou=0.7,
    conf=0.25
)

# ✅ Display results
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")

